In [ ]:
%pip install --quiet "google-adk[extensions]" "google-cloud-aiplatform[agent_engines,adk]"

In [2]:
import json
import os
import urllib.request

import vertexai

PROJECT_ID = "qwiklabs-gcp-04-c35f29416864"   # lab panel
LOCATION = "us-central1"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
vertexai.init(project=PROJECT_ID, location=LOCATION)

GEMINI_MODEL = "gemini-2.5-flash"                 # or the model the instructor names
CLAUDE_MODEL = "claude-sonnet-4-5@20250929"       # copy the exact ID from Model Garden > Claude

HEADERS = {"User-Agent": "adk-nmp-weather-agent (nealpetro@aol.com)"}  # NWS requires a User-Agent


def _get_json(url: str) -> dict:
    """GET a URL and return its JSON body as a dict."""
    request = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(request, timeout=15) as response:
        return json.load(response)

In [3]:
def get_weather(latitude: float, longitude: float) -> dict:
    """Get current conditions and the forecast from the National Weather Service.

    Args:
        latitude: Latitude of a US location, for example 38.9687.
        longitude: Longitude of a US location, for example -77.3411.

    Returns:
        A dict with "status" ("success" or "error"), "location", "current"
        (observed conditions at the nearest station, temperature in °F) and
        "forecast" (the next 3 forecast periods), or "error_message".
    """
    try:
        # NWS is two-step: /points tells us which forecast grid and stations cover the location.
        point = _get_json(f"https://api.weather.gov/points/{latitude:.4f},{longitude:.4f}")["properties"]
        periods = _get_json(point["forecast"])["properties"]["periods"][:3]
        station = _get_json(point["observationStations"])["features"][0]["properties"]["stationIdentifier"]
        observation = _get_json(f"https://api.weather.gov/stations/{station}/observations/latest")["properties"]
    except Exception as e:
        return {"status": "error", "error_message": f"National Weather Service request failed (US only): {e}"}

    temp_c = observation["temperature"]["value"]
    place = point["relativeLocation"]["properties"]
    return {
        "status": "success",
        "location": f"{place['city']}, {place['state']}",
        "current": {
            "station": station,
            "observed_at": observation["timestamp"],
            "conditions": observation["textDescription"],
            "temperature_f": round(temp_c * 9 / 5 + 32) if temp_c is not None else None,  # NWS reports °C
        },
        "forecast": [
            {"period": p["name"], "temperature": f"{p['temperature']} {p['temperatureUnit']}",
             "forecast": p["detailedForecast"]}
            for p in periods
        ],
        "source": "National Weather Service (api.weather.gov)",
    }

In [4]:
# Coordinates checked against api.weather.gov/points (each resolves to the right city).
CITY_COORDINATES = {
    "reston, va": (38.9687, -77.3411),
    "washington, dc": (38.8951, -77.0364),
    "arlington, va": (38.8816, -77.0910),
    "new york, ny": (40.7128, -74.0060),
    "chicago, il": (41.8781, -87.6298),
    "denver, co": (39.7392, -104.9903),
    "seattle, wa": (47.6062, -122.3321),
    "miami, fl": (25.7617, -80.1918),
    "austin, tx": (30.2672, -97.7431),
    "los angeles, ca": (34.0522, -118.2437),
}


def get_coordinates(city: str, state: str) -> dict:
    """Look up the latitude and longitude of a supported US city. Call this before get_weather.

    Args:
        city: The city name, for example "Reston".
        state: The two-letter state code, for example "VA".

    Returns:
        A dict with "status", "city", "state", "latitude" and "longitude",
        or "error_message" listing the supported cities.
    """
    key = f"{city.strip().lower()}, {state.strip().lower()}"
    if key not in CITY_COORDINATES:
        supported = ", ".join(name.title() for name in CITY_COORDINATES)
        return {"status": "error", "error_message": f"{city}, {state} is not supported. Supported cities: {supported}"}
    latitude, longitude = CITY_COORDINATES[key]
    return {"status": "success", "city": city.title(), "state": state.upper(),
            "latitude": latitude, "longitude": longitude}


In [5]:
from google.adk.agents import Agent
from google.adk.models.anthropic_llm import Claude

INSTRUCTION = (
    "You are a weather assistant for US locations.\n"
    "1. Call get_coordinates with the city and the two-letter state code.\n"
    "2. Call get_weather with the latitude and longitude it returns.\n"
    "3. Answer with current conditions first, then a short forecast.\n"
    "Only report what the tools return. If a tool returns an error, explain it plainly."
)


def build_weather_agent(model) -> Agent:
    """Build the weather agent on any ADK-supported model (a Gemini name or a model object)."""
    return Agent(
        name="weather_agent",
        model=model,
        description="Gives real-time weather for US locations using the National Weather Service.",
        instruction=INSTRUCTION,
        tools=[get_coordinates, get_weather],
    )


MODELS = {
    "gemini": GEMINI_MODEL,                  # Gemini: just the model name
    "claude": Claude(model=CLAUDE_MODEL),    # Claude on Vertex AI: same project, no Anthropic key
    # Other providers via LiteLLM (needs that provider's API key in an env var):
    # "gpt": LiteLlm(model="openai/gpt-4o"),   # from google.adk.models.lite_llm import LiteLlm
}
agents = {name: build_weather_agent(model) for name, model in MODELS.items()}


In [6]:
CITIES = ["Reston, VA", "Chicago, IL", "Denver, CO", "Seattle, WA", "Miami, FL"]
for city in CITIES:
    place = get_coordinates(*city.split(", "))
    assert place["status"] == "success", place
    weather = get_weather(place["latitude"], place["longitude"])
    assert weather["status"] == "success", weather
    print(f"PASS {city:12} -> {weather['location']:14} {weather['current']['temperature_f']}°F "
          f"{weather['current']['conditions']} | {weather['forecast'][0]['period']}: {weather['forecast'][0]['temperature']}")
assert get_coordinates("Gotham", "NY")["status"] == "error"   # not in the lookup table
print("PASS unknown city returns an error")
assert get_weather(51.5, -0.12)["status"] == "error"          # London: outside NWS coverage
print("PASS non-US location returns an error\nTOOL TESTS PASSED")


PASS Reston, VA   -> Reston, VA     64°F Clear | This Afternoon: 70 F
PASS Chicago, IL  -> Chicago, IL    61°F Partly Cloudy | Today: 60 F
PASS Denver, CO   -> Denver, CO     41°F Cloudy | Today: 56 F
PASS Seattle, WA  -> Seattle, WA    52°F Mostly Cloudy | Today: 55 F
PASS Miami, FL    -> Miami, FL      82°F Mostly Cloudy | Today: 82 F
PASS unknown city returns an error
PASS non-US location returns an error
TOOL TESTS PASSED


In [8]:
import logging
import threading
import warnings

from vertexai.preview import reasoning_engines

# Keep the test output readable: hide library warnings, and capture ADK's internal error
# logs so a failing model is reported as one clear line instead of a full traceback.
warnings.filterwarnings("ignore")
logging.getLogger("vertexai").setLevel(logging.ERROR)


class LastError(logging.Handler):
    """Remembers the most recent error ADK logged (e.g. a model that isn't available)."""
    message = ""

    def emit(self, record):
        if record.exc_info:
            self.message = f"{type(record.exc_info[1]).__name__}: {record.exc_info[1]}"[:300]


last_error = LastError()
adk_logger = logging.getLogger("google_adk")
adk_logger.addHandler(last_error)
adk_logger.propagate = False
threading.excepthook = lambda args: None  # the same error is re-raised in a worker thread; already captured


def ask(app, session_id: str, message: str) -> list[str]:
    """Send one message; print tool calls and the answer; return the tool names called."""
    calls = []
    print(f"\nYou: {message}")
    for event in app.stream_query(user_id="tester", session_id=session_id, message=message):
        for part in event.get("content", {}).get("parts", []):
            if "function_call" in part:
                calls.append(part["function_call"]["name"])
                print(f"  [calling {part['function_call']['name']}]")
            elif part.get("text"):
                print("Agent:", part["text"])
    return calls


for model_name, agent in agents.items():
    print(f"\n========== {model_name.upper()} ==========")
    try:
        app = reasoning_engines.AdkApp(agent=agent)
        session = app.create_session(user_id="tester")
        session_id = session.id if hasattr(session, "id") else session["id"]
        for city in CITIES[:3]:
            last_error.message = ""
            calls = ask(app, session_id, f"What's the weather like in {city}?")
            if not calls:
                raise RuntimeError(last_error.message or "the model returned no response")
            assert calls[:2] == ["get_coordinates", "get_weather"], calls
            print(f"PASS {model_name}: {city}")
    except Exception as e:
        # Keeps the notebook running if a model isn't enabled in this project; the output shows why.
        print(f"SKIPPED {model_name}: {e}")



========== GEMINI ==========

You: What's the weather like in Reston, VA?
  [calling get_coordinates]
  [calling get_weather]
Agent: The current weather in Reston, VA is clear with a temperature of 64°F.

The forecast is as follows:
*   **This Afternoon**: Sunny, with a high near 70. Northwest wind around 6 mph.
*   **Tonight**: Mostly clear, with a low around 47. Northwest wind 1 to 5 mph.
*   **Tuesday**: Sunny, with a high near 69. North wind around 6 mph.
PASS gemini: Reston, VA

You: What's the weather like in Chicago, IL?
  [calling get_coordinates]
  [calling get_weather]
Agent: The current weather in Chicago, IL is partly cloudy with a temperature of 63°F.

The forecast is as follows:
*   **Today**: Mostly cloudy. High near 60, with temperatures falling to around 58 in the afternoon. North northeast wind around 15 mph, with gusts as high as 25 mph.
*   **Tonight**: Mostly cloudy, with a low around 52. East northeast wind 10 to 15 mph, with gusts as high as 25 mph.
*   **Tuesda

Claude returned 404 in this lab project (not enabled / not offered in us-central1), so its test is skipped. With Claude enabled in Model Garden, set CLAUDE_MODEL to projects/PROJECT_ID/locations/REGION/publishers/anthropic/models/MODEL_ID (a region where Claude is offered, e.g. us-east5) and it should run without other code changes.